# Notebook 10 — Live Test

**Proyecto Final PLN — Agentic RAG Mundial 2026**

Notebook minimalista para **demo en vivo**. Construye el agente una vez y deja una celda al final donde escribir cualquier pregunta y verla responder con stream de los nodos.

## Como usar

1. **Run All** las celdas 1-2 una sola vez (carga Chroma + arma el agente). Toma ~5 segundos.
2. **Bajar al final**, editar la variable `pregunta` con lo que quieras preguntar y correr la celda.
3. Re-editar y re-correr la celda final cuantas veces quieras — el agente queda vivo en memoria.

## Que muestra el output de la celda final

Por cada nodo del grafo se imprime:

- `generate_query_or_respond` → si el LLM decidio llamar tools y cuales (con sus queries internas).
- `retrieve` → resumen de los chunks recuperados (primeros 400 chars).
- `generate_answer` → respuesta final del agente, citando fuentes.

## No reinicia el kernel

Cada pregunta reusa el grafo ya compilado. No paga el costo de cargar nada de nuevo.

## 1. Setup + construccion del agente

Una sola celda que carga todo. Correr una vez.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en .env"

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
TOP_K_TOOL = 5
TOOL_MAX_CHARS_PER_CHUNK = 800
RECURSION_LIMIT = 4

# LLM + embeddings
embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=os.getenv("OPENAI_API_KEY"))
llm = ChatOpenAI(model=OPENAI_MODEL, api_key=os.getenv("OPENAI_API_KEY"), temperature=0)

# Chroma collections
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
col_mundial = client.get_collection("mundial")
col_plataforma = client.get_collection("plataforma")


def retrieve_from(collection, query, k=TOP_K_TOOL):
    q_emb = embedder.embed_query(query)
    res = collection.query(query_embeddings=[q_emb], n_results=k,
                           include=["documents", "metadatas", "distances"])
    return [{"content": d, "metadata": m}
            for d, m in zip(res["documents"][0], res["metadatas"][0])]


def format_context(chunks, max_chars=TOOL_MAX_CHARS_PER_CHUNK):
    parts = []
    for c in chunks:
        src = c["metadata"].get("titulo", c["metadata"].get("ruta", "?"))
        body = c["content"]
        if len(body) > max_chars:
            body = body[:max_chars].rstrip() + " [...]"
        parts.append(f"[{src}]\n{body}")
    return "\n\n".join(parts)


@tool
def buscar_mundial(query: str) -> str:
    """Busca informacion sobre el Mundial 2026, Mundiales historicos, selecciones, jugadores,
    estadios, reglamento FIFA, reglas y calendario completo del Mundial 2026 (104 partidos
    numerados 1-104 con fecha, estadio, fase, equipos; FINAL = partido 104).
    """
    chunks = retrieve_from(col_mundial, query)
    return format_context(chunks) if chunks else "Sin resultados."


@tool
def buscar_plataforma(query: str) -> str:
    """Busca informacion sobre la plataforma 91 (noventayuno.com): como predecir, sistema de
    puntaje, rankings, tribus, torneos, monedas, tienda, perfil, referidos, T&C.
    """
    chunks = retrieve_from(col_plataforma, query)
    return format_context(chunks) if chunks else "Sin resultados."


TOOLS = [buscar_mundial, buscar_plataforma]

GENERATE_PROMPT = (
    "Eres un asistente de preguntas y respuestas en español sobre el Mundial 2026 y la plataforma 91 de predicciones.\n\n"
    "REGLAS:\n"
    "1. Usa UNICAMENTE la informacion del contexto. NO inventes datos.\n"
    "2. Si la pregunta tiene varias partes y solo encuentras info para algunas, responde lo que SI sabes Y declara 'No tengo informacion sobre [parte que falta]'.\n"
    "3. Si el contexto no contiene nada util, responde 'No tengo informacion sobre esto'.\n"
    "4. Cita la fuente entre corchetes [titulo].\n"
    "5. Conciso (max 4-5 oraciones). Responde en español.\n\n"
    "Pregunta: {question}\n\nContexto:\n{context}"
)


def generate_query_or_respond(state):
    return {"messages": [llm.bind_tools(TOOLS).invoke(state["messages"])]}


def generate_answer(state):
    messages = state["messages"]
    question = messages[0].content
    contexts = []
    for msg in reversed(messages):
        if hasattr(msg, "type") and msg.type == "tool":
            contexts.append(msg.content)
        elif contexts:
            break
    contexts.reverse()
    context = "\n\n---\n\n".join(contexts) if contexts else "(sin contexto)"
    return {"messages": [llm.invoke([{"role": "user", "content": GENERATE_PROMPT.format(question=question, context=context)}])]}


# Compilar grafo
workflow = StateGraph(MessagesState)
workflow.add_node("generate_query_or_respond", generate_query_or_respond)
workflow.add_node("retrieve", ToolNode(TOOLS))
workflow.add_node("generate_answer", generate_answer)
workflow.add_edge(START, "generate_query_or_respond")
workflow.add_conditional_edges("generate_query_or_respond", tools_condition, {"tools": "retrieve", END: END})
workflow.add_edge("retrieve", "generate_answer")
workflow.add_edge("generate_answer", END)
graph = workflow.compile()

print(f"Agente listo:")
print(f"  LLM:        {OPENAI_MODEL}")
print(f"  Embeddings: {EMBEDDING_MODEL}")
print(f"  Chroma:     mundial={col_mundial.count()} chunks, plataforma={col_plataforma.count()} chunks")
print(f"  Tools:      buscar_mundial, buscar_plataforma (k={TOP_K_TOOL})")
print("\nBajar a la celda final para hacer preguntas en vivo.")

Agente listo:
  LLM:        gpt-4o-mini
  Embeddings: text-embedding-3-small
  Chroma:     mundial=4497 chunks, plataforma=316 chunks
  Tools:      buscar_mundial, buscar_plataforma (k=5)

Bajar a la celda final para hacer preguntas en vivo.


## 2. Helper `correr(pregunta)`

Funcion minima — solo recibe la pregunta y imprime el paso a paso de los nodos del agente.

In [2]:
def correr(pregunta):
    print("=" * 78)
    print(f"PREGUNTA: {pregunta}")
    print("=" * 78)
    config = {"recursion_limit": RECURSION_LIMIT}
    for chunk in graph.stream({"messages": [{"role": "user", "content": pregunta}]}, config=config):
        for node, update in chunk.items():
            print(f"\n--- Nodo: {node} ---")
            msg = update["messages"][-1]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  → tool_call: {tc['name']}({tc['args']})")
            content = getattr(msg, "content", "")
            if content:
                if hasattr(msg, "type") and msg.type == "tool":
                    print(f"  [tool result, {len(content)} chars — primeros 400]\n  {content[:400]}...")
                else:
                    print(content)
    print()


print("Funcion 'correr(pregunta)' lista. Uso: correr('¿tu pregunta?')")

Funcion 'correr(pregunta)' lista. Uso: correr('¿tu pregunta?')


## 3. Ideas de preguntas (opcional, solo referencia)

Por si el profesor no sabe que preguntar, ejemplos cubriendo distintas categorias:

**Mundial 2026:**
- *¿Cuantas selecciones participan en el Mundial 2026?*
- *¿En que estadio se juega la final del Mundial 2026?*
- *¿Cuando juega Mexico su primer partido?*
- *¿Cuantas plazas tiene CONMEBOL en el Mundial 2026?*

**Mundiales historicos:**
- *¿Quien gano el Mundial 2022?*
- *¿En que pais se jugo el primer Mundial?*

**Reglamento FIFA:**
- *¿Que dice el reglamento sobre la prorroga y los penales?*
- *¿Cuantos puntos se descuentan por una tarjeta roja directa?*

**Plataforma 91:**
- *¿Cuantos puntos gano si acierto el marcador exacto?*
- *¿Como creo una tribu en la plataforma?*
- *¿Que pasa si invito amigos con mi referido?*

**Multi-hop (cruzan ambos corpus):**
- *¿En que estadio juega Argentina su primer partido y como predigo en la plataforma?*
- *¿Cuando es la final del Mundial 2026 y como hago la prediccion en la plataforma?*

**Conversacional / capacidades:**
- *Hola, ¿que puedes hacer?*

**Edge cases (puede no tener respuesta completa):**
- *¿Cuanto cuesta entrar al Mundial 2026?* (no esta en corpus → agente debe decir "no tengo informacion")

## 4. ⭐ CELDA EN VIVO — editar la pregunta y correr

Cambiar el texto entre comillas y dar Run a esta celda. Se puede re-correr infinitas veces.

In [3]:
pregunta = "¿En que estadio se juega la final del Mundial 2026?"

correr(pregunta)

PREGUNTA: ¿En que estadio se juega la final del Mundial 2026?

--- Nodo: generate_query_or_respond ---
  → tool_call: buscar_mundial({'query': 'final Mundial 2026 estadio'})

--- Nodo: retrieve ---
  [tool result, 3030 chars — primeros 400]
  [Estadio BBVA]
o con el arquitecto Federico Velasco,​​​ V&FO Arquitectos,​​ que ha trabajado en los diseños desde 2008.​ El estadio fue elegido para ser uno de los estadios sede de la Copa Mundial de Fútbol 2026.​​​Características
Tiene capacidad para 53 500 espectadores, después de un incremento a su aforo inicial de 51 000.​
324 Suites.
Cuenta con 2 restaurantes.
Zona comercial.
Tienda oficial d...

--- Nodo: generate_answer ---
La final del Mundial 2026 se disputará el domingo 19 de julio de 2026 a las 14:00 hora local en el MetLife Stadium de East Rutherford, NJ, USA [Final del Mundial 2026]. No tengo información sobre otros estadios que puedan albergar la final.



In [4]:
pregunta = "Que pasa si empato en el ranking entre tribus de 91?"

correr(pregunta)

PREGUNTA: Que pasa si empato en el ranking entre tribus de 91?

--- Nodo: generate_query_or_respond ---
  → tool_call: buscar_plataforma({'query': 'empate en el ranking entre tribus'})

--- Nodo: retrieve ---
  [tool result, 1119 chars — primeros 400]
  [Ranking Entre Tribus]
# Ranking Entre Tribus

El Ranking Entre Tribus muestra cómo compiten las tribus entre sí dentro de 91. Las tribus compiten por el promedio de puntos de sus miembros.

[Criterios de desempate en rankings]
## Desempate dentro de una tribu

Se aplican las mismas reglas del Ranking Global, pero solo entre miembros activos de la tribu.

[Torneos de tribus — cómo funcionan]
### 1...

--- Nodo: generate_answer ---
Si hay un empate en el ranking entre tribus de 91, se aplican los siguientes criterios de desempate: primero, se considera el mayor promedio de puntos de la tribu; luego, la mayor suma total de predicciones exactas entre todos los miembros; después, el mayor promedio de predicciones exactas por miembro; seguid

In [24]:
pregunta = "que pasa si gano un torneo"

correr(pregunta)

PREGUNTA: que pasa si gano un torneo

--- Nodo: generate_query_or_respond ---
  → tool_call: buscar_plataforma({'query': 'ganar un torneo'})

--- Nodo: retrieve ---
  [tool result, 1365 chars — primeros 400]
  [Torneos individuales — cómo funcionan]
## Flujo para inscribirse

1. Entrar a https://www.noventayuno.com/torneos
2. Revisar los torneos disponibles.
3. Elegir el torneo en el que se quiere participar.
4. Revisar premio, fecha de inicio y costo en monedas.
5. Inscribirse antes de que empiece el torneo.
6. Predecir partidos dentro de la ventana del torneo.
7. Revisar la posición en el ranking del ...

--- Nodo: generate_answer ---
Si ganas un torneo, recibirás el premio correspondiente al primer puesto. Por ejemplo, en el Torneo LEGO de la Copa del Mundo, el premio es un LEGO de la Copa del Mundo, y en el Torneo Johnnie Walker Blue Label, el premio es una botella de Johnnie Walker Blue Label. No tengo información sobre otros beneficios o consecuencias de ganar un torneo. [Torneos